# Ch9 — Grounded synthesis

Retrieval (Ch8) already produced the answer *values* through the graph. Synthesis turns those verified facts and their source spans into prose — **without** letting the model reach into its parametric memory. The `Assembler` hands the synthesis LLM an `<evidence>` block and a system prompt that says *answer ONLY from these*. When the evidence does not contain the answer, the model is instructed to decline.

This notebook is grounded in `data/corpus_facts.md` (Northwind Industries FY2025). Cells tagged **[GPU/Qwen — CI]** load Qwen and are executed by the lead in CI; every other cell runs on CPU with a scripted fake backend so the chapter's claim is checkable deterministically.

In [ ]:
import os, sys
KNOWLYTIX_SRC = os.environ.get("KNOWLYTIX_SRC", "/home/user/jupyterlab/GMS-knowlytix")
sys.path.insert(0, KNOWLYTIX_SRC)

In [ ]:
from knowlytix.knowledge.rag.assemble import Assembler
from knowlytix.knowledge.rag.retrieve import RetrievedFact
from knowlytix.knowledge.rag.config import RagConfig
from knowlytix.knowledge.llm_backend import LLMBackend
from knowlytix.knowledge.geode import QWEN_3B

print('synthesis model (real path):', QWEN_3B)

In [ ]:
# Facts as Ch8's Retriever would return them, with provenance spans.
# Source: data/corpus_facts.md sample retrieval for "cloud platform revenue".
facts = [
    RetrievedFact(
        head="cloud platform", relation="has_revenue", tail="120.0",
        score=0.0, confidence=1.0, source="triple",
        location=":15:553-558", raw="Cloud Platform | Technology | 120.0 | 340",
    ),
]
ENM_VALUE = "120.0"   # segment_performance / Cloud Platform/Technology/Revenue
print(Assembler._context("What was Cloud Platform revenue?", facts))

In [ ]:
class FakeBackend(LLMBackend):
    """Deterministic stand-in for Qwen: grounds in the evidence string.

    Mirrors a well-behaved synthesizer: it answers from the FACT line when
    the asked value is present, otherwise it declines. Used so the chapter's
    claim is checkable in CI without a GPU.
    """
    def __init__(self, model="fake-grounded"):
        self._model = model

    def call(self, system: str, user: str, max_tokens: int = 2048) -> str:
        if "120.0" in user:
            return "Cloud Platform reported revenue of 120.0."
        return "I cannot answer that from the available evidence."

    @property
    def model_name(self) -> str:
        return self._model

In [ ]:
assembler = Assembler(FakeBackend())
grounded = assembler.assemble("What was Cloud Platform revenue?", facts)
print(grounded)

In [ ]:
# [GPU/Qwen — CI] Real synthesis path. The lead runs this in CI; it is the
# per-role LLM choice (local Qwen2.5-3B-Instruct, no API key).
from knowlytix.knowledge.llm_backend import LocalTransformersBackend

qwen = LocalTransformersBackend(QWEN_3B)
real_answer = Assembler(qwen).assemble("What was Cloud Platform revenue?", facts)
print(real_answer)
# Expected: prose stating revenue of 120.0, no other figure.

In [ ]:
# No fact answers an Outlook question (a coverage blind spot, Ch11): the
# evidence block is empty, so a grounded synthesizer must decline.
no_facts: list[RetrievedFact] = []
refusal = Assembler(FakeBackend()).assemble(
    "What does the Outlook section forecast for FY2026?", no_facts)
print(refusal)

## Exercise — per-role synthesis LLM
`RagConfig` keeps the three LLM roles separate: `llm` (synthesis), `llm_extract` (NL -> query triples), `llm_verify` (answer -> claim triples). Swap *only* the synthesis backend and confirm the other roles fall back per `RagConfig.extract_llm()` / `verify_llm()`.

In [ ]:
extract_be = FakeBackend("fake-extract")
synth_be = FakeBackend("fake-synth")

cfg = RagConfig(llm=synth_be, llm_extract=extract_be)
# Synthesis uses the swapped backend; verify defaults to the extract role.
assert cfg.llm.model_name == "fake-synth"
assert cfg.extract_llm().model_name == "fake-extract"
assert cfg.verify_llm().model_name == "fake-extract"   # verify -> extract
print("synthesis :", cfg.llm.model_name)
print("extract   :", cfg.extract_llm().model_name)
print("verify    :", cfg.verify_llm().model_name)

## Self-check
The chapter's claim: a grounded answer **contains the ENM figure and no other number**, and the synthesizer **refuses** when the evidence lacks the answer.

In [ ]:
import re

# (a) the grounded answer carries the exact ENM figure ...
assert ENM_VALUE in grounded
# ... and introduces no other number.
nums = re.findall(r"\d+(?:\.\d+)?", grounded)
assert set(nums) == {ENM_VALUE}, nums
# (b) with no supporting facts, the synthesizer declines.
assert "cannot answer" in refusal.lower()
print("Ch9 self-check passed: grounded synthesis carries only the ENM figure;"
      " empty evidence -> refusal.")